<a href="https://colab.research.google.com/github/mannangrover/Diabities_prediction_system/blob/main/WearableDiabaties.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Update and install deps

In [ ]:
!pip install kagglehub




Downloading Dataset

In [ ]:
# Install dependencies if needed:
# !pip install kagglehub[hf-datasets]

import kagglehub
from kagglehub import KaggleDatasetAdapter
from google.colab import userdata
import os
import shutil  # Library for moving files

# --- 1. Authentication ---
os.environ['KAGGLE_USERNAME'] = userdata.get('kaggle_username')
# You called your secret 'kaggle_token', so we use that here:
os.environ['KAGGLE_KEY'] = userdata.get('kaggle_token')

# --- 2. Download ---
print("Downloading dataset...")
dataset_path = kagglehub.dataset_download("dariushbahrami/cdc-brfss-survey-2021")
print(f"Cache location: {dataset_path}")

# --- 3. Identify and Move File to Home ---
# Based on your previous error, we know the real file name is LLCP2021.csv
real_filename = "LLCP2021.csv"
source_path = os.path.join(dataset_path, real_filename)
destination_path = f"/content/{real_filename}" # /content/ is "Home" in Colab

if os.path.exists(source_path):
    # Copy the file to the main folder
    shutil.copy(source_path, destination_path)
    print(f"\nSUCCESS: File copied to your home folder: {destination_path}")

    # --- 4. Load the Dataset ---
    print("Loading into Hugging Face Dataset...")
    hf_dataset = kagglehub.load_dataset(
      KaggleDatasetAdapter.HUGGING_FACE,
      "dariushbahrami/cdc-brfss-survey-2021",
      real_filename,
    )
    print("Dataset loaded successfully!")
    print(hf_dataset)

else:
    print(f"\nError: Could not find '{real_filename}' in the downloaded folder.")
    print("Files found:", os.listdir(dataset_path))

100%|██████████| 48.2M/48.2M [00:00<00:00, 53.2MB/s]

Extracting files...


Cache location: /root/.cache/kagglehub/datasets/dariushbahrami/cdc-brfss-survey-2021/versions/1

SUCCESS: File copied to your home folder: /content/LLCP2021.csv
Loading into Hugging Face Dataset...


/tmp/ipython-input-504534102.py:33: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  hf_dataset = kagglehub.load_dataset(


Using Colab cache for faster access to the 'cdc-brfss-survey-2021' dataset.
Dataset loaded successfully!
Dataset({
    features: ['_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_PSU', 'CTELENM1', 'PVTRESD1', 'COLGHOUS', 'STATERE1', 'CELPHON1', 'LADULT1', 'COLGSEX', 'NUMADULT', 'LANDSEX', 'NUMMEN', 'NUMWOMEN', 'RESPSLCT', 'SAFETIME', 'CTELNUM1', 'CELLFON5', 'CADULT1', 'CELLSEX', 'PVTRESD3', 'CCLGHOUS', 'CSTATE1', 'LANDLINE', 'HHADULT', 'SEXVAR', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'PRIMINSR', 'PERSDOC3', 'MEDCOST1', 'CHECKUP1', 'EXERANY2', 'BPHIGH6', 'BPMEDS', 'CHOLCHK3', 'TOLDHI3', 'CHOLMED3', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'ASTHMA3', 'ASTHNOW', 'CHCSCNCR', 'CHCOCNCR', 'CHCCOPD3', 'ADDEPEV3', 'CHCKDNY2', 'DIABETE4', 'DIABAGE3', 'HAVARTH5', 'ARTHEXER', 'ARTHEDU', 'LMTJOIN3', 'ARTHDIS2', 'JOINPAI2', 'MARITAL', 'EDUCA', 'RENTHOM1', 'NUMHHOL3', 'NUMPHON3', 'CPDEMO1B', 'VETERAN3', 'EMPLOY1', 'CHILDREN', 'INCOME3', 'PREGNANT', 'WEIGHT2', 'HEIGHT

Cleaning the data and sorting a few metric that we might need

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the Dataset
# ---------------------------------------------------------
# Ensure 'LLCP2021.csv' is present in the local directory.
file_path = '/content/LLCP2021.csv'
print(f"Initiating data load from: {file_path}")
try:
    df = pd.read_csv(file_path)
    print(f"Data loaded successfully. Initial Dimensions: {df.shape}")
except FileNotFoundError:
    print("Error: File not found. Please upload LLCP2021.csv.")
    raise

# 2. Define Feature Subset
# ---------------------------------------------------------
# REMOVED: INCOME3, EDUCA
# RETAINED: Physiological, Activity, and Medical History metrics.
selected_features = [
    'DIABETE4',   # Target Variable
    '_BMI5',      # Body Mass Index
    'EXERANY2',   # Exercise Status (Last 30 Days)
    '_TOTINDA',   # Physical Activity Index
    'DIFFWALK',   # Difficulty Walking
    'GENHLTH',    # General Health Rating
    '_PHYS14D',   # Physical Health (Days not good)
    'BPMEDS',     # Blood Pressure Medication Use
    'BPHIGH6',    # History of High Blood Pressure
    'TOLDHI3',    # History of High Cholesterol
    '_MICHD',     # History of Coronary Heart Disease/MI
    'SMOKE100',   # Lifetime Smoking Status
    'ALCDAY5',    # Alcohol Consumption Frequency
    '_FRUTSU1',   # Daily Fruit Consumption
    '_VEGESU1',   # Daily Vegetable Consumption
    '_AGEG5YR',   # Age Category
    'SEXVAR'      # Biological Sex
]

# Create a filtered dataframe to avoid SettingWithCopy warnings
df_clean = df[selected_features].copy()

# 3. Target Variable Standardization (DIABETE4)
# ---------------------------------------------------------
# Logic:
# 0 = No Diabetes (Codes 2, 3)
# 1 = Diabetes or Pre-diabetes (Codes 1, 4)
# Exclude: 7 (Don't know), 9 (Refused)

df_clean = df_clean[~df_clean['DIABETE4'].isin([7, 9])]
df_clean['DIABETE4'] = df_clean['DIABETE4'].replace({
    2: 0, # Gestational diabetes -> No
    3: 0, # No -> No
    1: 1, # Yes -> Yes
    4: 1  # Pre-diabetes -> Yes
})

# 4. Feature Engineering and Cleaning
# ---------------------------------------------------------

# --- _BMI5 (Body Mass Index) ---
# Code 9999 denotes missing. Values contain 2 implied decimal places (e.g., 2500 = 25.00).
df_clean['_BMI5'] = df_clean['_BMI5'].replace(9999, np.nan) / 100

# --- ALCDAY5 (Alcohol Frequency) ---
# Code 888 = 0 drinks.
# Codes 101-107 = Days per week.
# Codes 201-230 = Days per month.
# Codes 777/999 = Missing.
def clean_alcohol_data(value):
    if value == 888:
        return 0
    elif 101 <= value <= 107:
        return (value - 100) * 4.3  # Convert weekly frequency to monthly approximation
    elif 201 <= value <= 230:
        return value - 200          # Already monthly frequency
    else:
        return np.nan               # Treat refusal/don't know as missing

df_clean['ALCDAY5'] = df_clean['ALCDAY5'].apply(clean_alcohol_data)

# --- _FRUTSU1 & _VEGESU1 (Dietary Intake) ---
# Values contain 2 implied decimal places. Code 9999 denotes missing.
df_clean['_FRUTSU1'] = df_clean['_FRUTSU1'].replace(9999, np.nan) / 100
df_clean['_VEGESU1'] = df_clean['_VEGESU1'].replace(9999, np.nan) / 100

# --- _TOTINDA (Physical Activity Index) ---
# 1 = Active, 2 = Inactive. Map 2 to 0 for binary boolean logic.
df_clean['_TOTINDA'] = df_clean['_TOTINDA'].replace({2: 0, 9: np.nan})

# --- GENHLTH (General Health) ---
# Scale 1-5. Remove 7/9 (Refused/Missing).
df_clean['GENHLTH'] = df_clean['GENHLTH'].replace([7, 9], np.nan)

# --- _PHYS14D (Physical Health Status) ---
# Remove 9 (Missing).
df_clean['_PHYS14D'] = df_clean['_PHYS14D'].replace(9, np.nan)

# --- Binary Variable Normalization ---
# For columns where 1=Yes, 2=No: Map 2 to 0. Remove 7/9.
binary_cols = ['EXERANY2', 'DIFFWALK', 'SMOKE100', '_MICHD', 'BPMEDS', 'BPHIGH6', 'TOLDHI3']

# Note regarding TOLDHI3: Dataset sometimes uses 1=Yes, 2=No.
# Verify specific year codebook if strict adherence is required, but standard logic applies here.
for col in binary_cols:
    df_clean[col] = df_clean[col].replace({2: 0, 7: np.nan, 9: np.nan})

# --- Demographics ---
# _AGEG5YR: 14 denotes missing/refused.
df_clean['_AGEG5YR'] = df_clean['_AGEG5YR'].replace(14, np.nan)

# 5. Missing Value Handling
# ---------------------------------------------------------
# Drop any row containing NaN values to ensure dataset integrity.
df_clean = df_clean.dropna()
print(f"Dimensions after cleaning missing values: {df_clean.shape}")

# 6. Class Balancing (Undersampling)
# ---------------------------------------------------------
# To prevent model bias towards the majority class (Non-Diabetic),
# we undersample the negative class to match the positive class count.

diabetic_records = df_clean[df_clean['DIABETE4'] == 1]
non_diabetic_records = df_clean[df_clean['DIABETE4'] == 0]

# Sample from non-diabetic records
non_diabetic_downsampled = non_diabetic_records.sample(n=len(diabetic_records), random_state=42)

# Concatenate and shuffle
df_balanced = pd.concat([diabetic_records, non_diabetic_downsampled])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Final Balanced Dataset Dimensions: {df_balanced.shape}")
print(f"Class Distribution (DIABETE4):\n{df_balanced['DIABETE4'].value_counts()}")

# 7. File Export
# ---------------------------------------------------------
output_filename = '/content/diabetes_binary_5050split_wearable_prototype.csv'
df_balanced.to_csv(output_filename, index=False)
print(f"Processing complete. Data exported to: {output_filename}")

Initiating data load from: /content/LLCP2021.csv
Data loaded successfully. Initial Dimensions: (438693, 303)
Dimensions after cleaning missing values: (119485, 17)
Final Balanced Dataset Dimensions: (67234, 17)
Class Distribution (DIABETE4):
DIABETE4
1.0    33617
0.0    33617
Name: count, dtype: int64
Processing complete. Data exported to: /content/diabetes_binary_5050split_wearable_prototype.csv


### **Target Variable (The Output)**

* **`DIABETE4`**
* **Meaning:** Does the user have Diabetes or Pre-diabetes?
* **Logic:** `0` = Healthy, `1` = Diabetes or Pre-diabetes. (Refusals removed).



---

### **Wearable & Physical Signals (The "Watch" Data)**

* **`_BMI5`**
* **Meaning:** Body Mass Index (calculated from height/weight).
* **Formula:** `Original Value / 100` (e.g., 2500 becomes 25.0).


* **`EXERANY2`**
* **Meaning:** Did they do *any* exercise in the last 30 days?
* **Logic:** `1` = Yes, `0` = No.


* **`_TOTINDA`**
* **Meaning:** Physical Activity Index (Are they "Active" vs. "Sedentary"?).
* **Logic:** `1` = Active, `0` = Inactive.


* **`DIFFWALK`**
* **Meaning:** Do they have serious difficulty walking or climbing stairs?
* **Logic:** `1` = Yes, `0` = No.


* **`_PHYS14D`**
* **Meaning:** How many days was their physical health "not good" this month?
* **Logic:** `1` = Zero days (Good), `2` = 1-13 days (Okay), `3` = 14+ days (Bad).



---

### **Medical History (User Profile)**

* **`BPMEDS`**
* **Meaning:** Are they currently taking Blood Pressure medication?
* **Logic:** `1` = Yes, `0` = No.


* **`BPHIGH6`**
* **Meaning:** Have they ever been told they have High Blood Pressure?
* **Logic:** `1` = Yes, `0` = No.


* **`TOLDHI3`**
* **Meaning:** Have they ever been told they have High Cholesterol?
* **Logic:** `1` = Yes, `0` = No.


* **`_MICHD`**
* **Meaning:** Heart Disease History (Heart Attack or Angina).
* **Logic:** `1` = Yes, `0` = No.


* **`GENHLTH`**
* **Meaning:** User's rating of their own health (1=Excellent to 5=Poor).
* **Logic:** Kept as 1-5 scale (Removed 7/9 refusals).



---

### **Habits & Demographics**

* **`_FRUTSU1`**
* **Meaning:** Fruit "Diet Score" (Times per day).
* **Formula:** `Original Value / 100` (e.g., 200 = 2.0 times/day).


* **`_VEGESU1`**
* **Meaning:** Vegetable "Diet Score" (Times per day).
* **Formula:** `Original Value / 100`.


* **`ALCDAY5`**
* **Meaning:** Alcohol frequency (Drinks per Month).
* **Formula:** Weekly values multiplied by `4.3` to get monthly; `888` (None) set to `0`.


* **`SMOKE100`**
* **Meaning:** Lifetime Smoker? (Smoked >100 cigs in entire life).
* **Logic:** `1` = Yes, `0` = No.


* **`_AGEG5YR`**
* **Meaning:** Age Bracket (1 = Age 18-24 ... 13 = Age 80+).
* **Logic:** 5-year increments.


* **`SEXVAR`**
* **Meaning:** Biological Sex.
* **Logic:** `1` = Male, `2` = Female.

Training using XGBoost with GPU runtime (make sure u select)

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import time

# 1. Configuration and Data Loading
# ---------------------------------------------------------
INPUT_FILE = '/content/diabetes_binary_5050split_wearable_prototype.csv'
MODEL_FILE = '/content/diabetes_wearable_prototype.json'

print(f"Loading dataset from {INPUT_FILE}...")
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"loaded Shape: {df.shape}")
except FileNotFoundError:
    print(f"ERROR: File {INPUT_FILE} not found. clean first")
    raise

# 2. Preprocessing
# ---------------------------------------------------------
# Separate Features (X) and Target (y)
target_col = 'DIABETE4'
X = df.drop(columns=[target_col])
y = df[target_col]

# Split into Training and Testing sets (80% Train, 20% Test)
# Stratify ensures the class balance (50/50) is maintained in the split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training Data: {X_train.shape}")
print(f"Testing Data:  {X_test.shape}")

# 3. Model Initialization (GPU Accelerated)
# ---------------------------------------------------------
# We use the XGBClassifier with 'gpu_hist' tree method.
# This offloads the heavy decision tree construction to the GPU.

print("\nXGBoost starting...")

model = xgb.XGBClassifier(
    # Core GPU Parameters
    device='cuda',            # Force CUDA usage
    tree_method='hist',       # Use histogram-based algorithm (required for efficient GPU usage)

    # Model Hyperparameters (Optimized for prototype)
    n_estimators=500,         # Number of trees
    learning_rate=0.05,       # Step size shrinkage
    max_depth=6,              # Maximum depth of a tree
    subsample=0.8,            # Subsample ratio of the training instances
    colsample_bytree=0.8,     # Subsample ratio of columns when constructing each tree

    # Objective
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

# 4. Training
# ---------------------------------------------------------
print("Starting training process...")
start_time = time.time()

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False  # Set to True to see real-time logloss reduction
)

end_time = time.time()
print(f"Training complete in {end_time - start_time:.2f} seconds.")

# 5. Evaluation
# ---------------------------------------------------------
print("\n--- Model Performance Metrics ---")

# Generate predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate metrics
acc = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(class_report)

print("Confusion Matrix:")
print(conf_matrix)

# 6. Feature Importance Analysis
# ---------------------------------------------------------
# This confirms which wearable signals are actually driving the predictions.
importance = model.feature_importances_
feature_names = X.columns
feat_imp = pd.DataFrame({'Feature': feature_names, 'Importance': importance})
feat_imp = feat_imp.sort_values(by='Importance', ascending=False)

print("\n--- 10 Most Critical Features ---")
print(feat_imp.head(10))

# 7. Model Export
# ---------------------------------------------------------
# Save the model in JSON format (efficient and portable)
model.save_model(MODEL_FILE)
print(f"\nModel saved to: {MODEL_FILE}")
print("load model using model.load_model()")

Loading dataset from /content/diabetes_binary_5050split_wearable_prototype.csv...
loaded Shape: (67234, 17)
Training Data: (53787, 16)
Testing Data:  (13447, 16)

XGBoost starting...
Starting training process...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [15:09:25] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training complete in 1.94 seconds.

--- Model Performance Metrics ---
Accuracy: 0.6723

Classification Report:
              precision    recall  f1-score   support

         0.0       0.68      0.65      0.66      6724
         1.0       0.66      0.70      0.68      6723

    accuracy                           0.67     13447
   macro avg       0.67      0.67      0.67     13447
weighted avg       0.67      0.67      0.67     13447

Confusion Matrix:
[[4340 2384]
 [2023 4700]]

--- 10 Most Critical Features ---
     Feature  Importance
4    GENHLTH    0.200488
6     BPMEDS    0.128774
8    TOLDHI3    0.126178
11   ALCDAY5    0.079259
3   DIFFWALK    0.076470
9     _MICHD    0.055982
0      _BMI5    0.051682
14  _AGEG5YR    0.048531
15    SEXVAR    0.044694
2   _TOTINDA    0.040207

Model saved to: /content/diabetes_wearable_prototype.json
load model using model.load_model()


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [15:09:27] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Tuning using multiple techniques bit bang

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

# 1. Define the parameter grid (The settings to test)
param_dist = {
    'learning_rate': uniform(0.01, 0.2),    # Step size
    'max_depth': randint(3, 10),            # Tree depth (prevent overfitting)
    'n_estimators': randint(100, 1000),     # Number of trees
    'subsample': uniform(0.6, 0.4),         # % of data used per tree
    'colsample_bytree': uniform(0.6, 0.4),  # % of features used per tree
    'gamma': uniform(0, 0.5)                # Min loss reduction required
}

# 2. Setup the Search
clf = xgb.XGBClassifier(
    device='cuda',
    tree_method='hist',
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

random_search = RandomizedSearchCV(
    clf,
    param_distributions=param_dist,
    n_iter=25,              # Try 25 different combinations
    scoring='recall',       # Optimize for RECALL (Catching more diabetics)
    cv=3,                   # 3-fold cross-validation
    verbose=1,
    n_jobs=1                # XGBoost handles parallelism internally
)

print("Starting Hyperparameter Tuning (Optimizing for Recall)...")
random_search.fit(X_train, y_train)

print(f"\nBest Recall Score: {random_search.best_score_:.4f}")
print("Best Parameters Found:")
print(random_search.best_params_)

# 3. Test the Best Model
best_model = random_search.best_estimator_
y_pred_optimized = best_model.predict(X_test)

print("\n--- Optimized Classification Report ---")
print(classification_report(y_test, y_pred_optimized))

Starting Hyperparameter Tuning (Optimizing for Recall)...
Fitting 3 folds for each of 25 candidates, totalling 75 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [15:09:27] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [15:09:27] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [15:09:28] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [15:09:28] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [15:09:30] WARNING: /w


Best Recall Score: 0.7108
Best Parameters Found:
{'colsample_bytree': np.float64(0.9844832019520633), 'gamma': np.float64(0.06029020057306955), 'learning_rate': np.float64(0.028722015347849142), 'max_depth': 5, 'n_estimators': 192, 'subsample': np.float64(0.8313215605412321)}

--- Optimized Classification Report ---
              precision    recall  f1-score   support

         0.0       0.69      0.65      0.67      6724
         1.0       0.67      0.70      0.68      6723

    accuracy                           0.68     13447
   macro avg       0.68      0.68      0.68     13447
weighted avg       0.68      0.68      0.68     13447



Found the best parameters using hyper tuning bit banging. Training new model based on found best parameters

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# 1. Configuration
# ---------------------------------------------------------
INPUT_FILE = '/content/diabetes_binary_5050split_wearable_prototype.csv'
MODEL_FILE = '/content/diabetes_wearable_final.json'

# 2. Load Data
# ---------------------------------------------------------
df = pd.read_csv(INPUT_FILE)
X = df.drop(columns=['DIABETE4'])
y = df['DIABETE4']

# 3. Apply Optimal Parameters
# ---------------------------------------------------------
# These are the exact values output by your RandomizedSearchCV
best_params = {
    'colsample_bytree': 0.8080409037504724,
    'gamma': 0.2168148501045089,
    'learning_rate': 0.05533870666155276,
    'max_depth': 4,
    'n_estimators': 478,
    'subsample': 0.7592918880823795,

    # System parameters
    'device': 'cuda',
    'tree_method': 'hist',
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'random_state': 42
}

print(f"Training Final Model with {best_params['n_estimators']} trees...")

# 4. Train on FULL Dataset (Optional but recommended for Production)
# ---------------------------------------------------------
# In production, we typically use all available data to maximize performance.
# However, to verify metrics one last time, we will keep the split.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

# 5. Final Verification
# ---------------------------------------------------------
y_pred = final_model.predict(X_test)
print("\n--- Final Production Metrics ---")
print(classification_report(y_test, y_pred))

# 6. Save Artifacts
# ---------------------------------------------------------
# Save the model architecture and weights
final_model.save_model(MODEL_FILE)
print(f"Model saved to: {MODEL_FILE}")

# Save the feature names order.
# CRITICAL: The app MUST send data in this exact order.
with open('/content/feature_names.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)
print("Feature order saved to: /content/feature_names.pkl")

Training Final Model with 478 trees...

--- Final Production Metrics ---
              precision    recall  f1-score   support

         0.0       0.69      0.65      0.67      6724
         1.0       0.67      0.71      0.69      6723

    accuracy                           0.68     13447
   macro avg       0.68      0.68      0.68     13447
weighted avg       0.68      0.68      0.68     13447

Model saved to: /content/diabetes_wearable_final.json
Feature order saved to: /content/feature_names.pkl
